In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder

from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier

import optuna

import shap

from scipy.stats import ks_2samp

import category_encoders as ce

import joblib

import warnings
warnings.filterwarnings('ignore')

In [5]:
class KingaMetricCreditRiskModel:
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.feature_names = None
        self.leaky_features = ['Payment_Behaviour', 'Delay_from_due_date', 'Num_of_Delayed_Payment']
    
    def load_and_preprocess(self, filepath):
        df = pd.read_csv(filepath)
    
        df = df.drop(columns=[c for c in self.leaky_features if c in df.columns], errors='ignore')

        print('Dropped leaky features:', [c for c in self.leaky_features if c in df])

        df = self.add_interaction_features(df) 

        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        """for col in df.select_dtypes('number').columns:
            Q1, Q3 = df[col].quantile([0.25, 0.75])
            IQR = Q3 - Q1
            df[col] = df[col].clip(lower=Q1-1.5*IQR, upper=Q3+1.5*IQR)"""

        df.fillna(df.median(numeric_only=True), inplace=True)
        
        X = df.drop('Default_Flag', axis=1)
        y = df['Default_Flag']

        self.feature_names = X.columns.tolist()
        
        return X, y
    
    def target_encode(self, X, y=None, fit=False):
        cat_cols = [c for c in ["Payment_of_Min_Amount", "Credit_Mix", "Borrower_Tier"] if c in X.columns]

        if fit:
            encoder = ce.TargetEncoder(cols=cat_cols)
            X_encoded = encoder.fit_transform(X, y)
            self.target_encoder = encoder
        else:
            if not hasattr(self, "target_encoder"):
                raise ValueError("Target encoder not fitted. Call with fit=True first.")
            X_encoded = self.target_encoder.transform(X)
        return X_encoded
    
    def add_interaction_features(self, df):
        df = df.copy()

        df["Debt_Stress"] = df["normalized_dti"] * df["normalized_utilization"]
        df["Repayment_Stress"] = df["normalized_emi"] * df["normalized_delinquency"]
        df["Liquidity_Index"] = df["normalized_savings"] * df["normalized_emi"]
        df["Credit_Exposure"] = df["Num_Credit_Card"] * df["Credit_Utilization_Ratio"]
        df["Risk_Index"] = (
            df["normalized_dti"] + 
            df["normalized_utilization"] + 
            df["normalized_delinquency"]
        ) / 3

        return df
    
    def feature_selection(self, X, y):

        folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        importances = pd.Series(0, index=X.columns)

        for train_idx, val_idx in folds.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]

            model = XGBClassifier(n_estimators=300, max_depth=4, random_state=42)
            model.fit(X_train, y_train)

            importances += pd.Series(model.feature_importances_, index=X.columns)
        
        importances /= folds.n_splits 

        top_features = importances.sort_values(ascending=False).head(25).index

        print("Top Features: ", top_features.tolist())

        return X[top_features], top_features
    
    def objective(self, trial, X_temp, y_temp, folds):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 300, 800),
            'max_depth': trial.suggest_int('max_depth', 3, 5),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08),
            'subsample': trial.suggest_float('subsample', 0.7, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.9),
            'min_child_weight': trial.suggest_int('min_child_weight', 5, 15),
            'gamma': trial.suggest_float('gamma', 0.1, 0.5),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 1),
            'reg_lambda': trial.suggest_float('reg_lambda', 1, 5),
            'random_state': 42,
            'tree_method': 'hist',
            'eval_metric': 'auc'
        }
        
        auc_scores = []

        for train_idx, val_idx in folds.split(X_temp, y_temp):
            X_train, X_val = X_temp.iloc[train_idx].copy(), X_temp.iloc[val_idx].copy()
            y_train, y_val = y_temp.iloc[train_idx], y_temp.iloc[val_idx]

            pos = (y_temp ==1).sum()
            neg = (y_temp ==0).sum()

            scale_pos_weight = neg/ (pos + 1e-6)
            params['scale_pos_weight'] = scale_pos_weight
            
            model = XGBClassifier(**params)
            model.fit(X_train, y_train)
            y_pred = model.predict_proba(X_val)[:,1]
            auc_scores.append(roc_auc_score(y_val, y_pred))
        
        return np.mean(auc_scores)
    
    def tune_hyperparams(self, X_temp, y_temp):
        folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        study = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner(n_startup_trials=10))
        
        study.optimize(lambda trial: self.objective(trial, X_temp, y_temp, folds), n_trials=200)
        
        return study.best_params
    
    def optimize_threshold(self, y_true, y_probs):

        thresholds = np.linspace(0.1, 0.9, 50)
        best_ks = 0
        best_thresh = 0.5

        for t in thresholds:
            preds = (y_probs >= t).astype(int)

            tpr = ((preds ==1) & (y_true == 1)).sum() / (y_true ==1).sum()
            fpr = ((preds ==1) & (y_true == 0)).sum() / (y_true ==0).sum()

            ks = tpr - fpr 

            if ks > best_ks:
                best_ks = ks
                best_thresh = t 

        self.best_threshold = best_thresh

        return best_thresh
    
    def train(self, filepath):
        print('Loading data...')
        X, y = self.load_and_preprocess(filepath)
        
        print('Splitting...')
        X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

        X_temp = self.target_encode(X_temp, y_temp, fit=True)
        X_test = self.target_encode(X_test, fit=False)

        print("Feature Selection...")
        X_temp, selected_features = self.feature_selection(X_temp, y_temp)
        X_test = X_test[selected_features]

        self.feature_names = selected_features
        
        print('Tuning hyperparams...')
        best_params = self.tune_hyperparams(X_temp, y_temp)
        print('Best params:', best_params)

        X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=42)

        self.model = XGBClassifier(**best_params, early_stopping_rounds=50, random_state=42, tree_method="hist", grow_policy="lossguide", max_leaves=32)

        self.model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        y_probs = self.model.predict_proba(X_val)[:,1]
        best_thresh = self.optimize_threshold(y_val, y_probs)

        print(f"OPtimal Threshold: {best_thresh:.3f}")        
        test_auc = roc_auc_score(y_test, self.model.predict_proba(X_test)[:,1])
        
        print(f'Test AUC: {test_auc:.4f}')
        
        if test_auc >= 0.85:
            print('🎉 Target AUC 0.85 achieved!')
        else:
            print('Target not met, consider ensemble/more features.')
        
        return test_auc
    
    def save(self, path='kinga_model.pkl'):
        joblib.dump(self, path)
        
        print(f'Model saved to {path}')

    def predict_proba(self, X_new):
        X_new = self.add_interaction_features(X_new)

        X_new = self.target_encoder.transform(X_new)

        X_new = X_new[self.feature_names]
        
        return self.model.predict_proba(X_new)[:,1]
        

if __name__ == '__main__':
    model = KingaMetricCreditRiskModel()
    auc = model.train('datasets/kingametric_credit_risk.csv')
    
    model.save()

Loading data...
Dropped leaky features: []
Splitting...
Feature Selection...


[I 2026-03-24 09:19:10,168] A new study created in memory with name: no-name-a27c87f4-cc52-4d84-ab92-b1222bd6e256


Top Features:  ['Borrower_Tier', 'normalized_delinquency', 'Risk_Index', 'normalized_emi', 'Payment_Instability', 'Num_of_Loan', 'Repayment_Stress', 'Total_EMI_per_month', 'Liquidity_Index', 'population_density_factor', 'Changed_Credit_Limit', 'Monthly_Inhand_Salary', 'Num_Credit_Inquiries', 'Annual_Income', 'Amount_invested_monthly', 'Credit_Depth', 'Net_Cash_Flow', 'Liquidity_Buffer', 'normalized_dti', 'Monthly_Balance', 'normalized_savings_capacity_ratio', 'Credit_Utilization_Ratio', 'Outstanding_Debt', 'normalized_savings', 'Credit_Exposure']
Tuning hyperparams...


[I 2026-03-24 09:19:11,997] Trial 0 finished with value: 0.6162996978442729 and parameters: {'n_estimators': 369, 'max_depth': 4, 'learning_rate': 0.06868688960681298, 'subsample': 0.7369319017579923, 'colsample_bytree': 0.7152040158618261, 'min_child_weight': 5, 'gamma': 0.379651702096226, 'reg_alpha': 0.3324313289452604, 'reg_lambda': 2.607182657812774}. Best is trial 0 with value: 0.6162996978442729.
[I 2026-03-24 09:19:13,298] Trial 1 finished with value: 0.6198713311435985 and parameters: {'n_estimators': 318, 'max_depth': 4, 'learning_rate': 0.07356222538155632, 'subsample': 0.8015941574409053, 'colsample_bytree': 0.7459179307501841, 'min_child_weight': 8, 'gamma': 0.14107110290974778, 'reg_alpha': 0.15460659644033226, 'reg_lambda': 4.766775260176178}. Best is trial 1 with value: 0.6198713311435985.
[I 2026-03-24 09:19:16,739] Trial 2 finished with value: 0.6284125722964186 and parameters: {'n_estimators': 435, 'max_depth': 5, 'learning_rate': 0.022714285246848157, 'subsample': 0

Best params: {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.0201317346333623, 'subsample': 0.7860241536052623, 'colsample_bytree': 0.8363102158954961, 'min_child_weight': 6, 'gamma': 0.2121459711628389, 'reg_alpha': 0.9308124567707152, 'reg_lambda': 4.42802956191281}
OPtimal Threshold: 0.312
Test AUC: 0.6333
Target not met, consider ensemble/more features.
Model saved to kinga_model.pkl


In [6]:
class KingaMetricCreditRiskModel:
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.feature_names = None
        self.leaky_features = ['Payment_Behaviour', 'Delay_from_due_date', 'Num_of_Delayed_Payment']
    
    def load_and_preprocess(self, filepath):
        df = pd.read_csv(filepath)

        #print(f'Dataset shape: {df.shape}, Default rate: {df["Default_Flag"].mean():.3f}')
        
        
        # Drop leaky features (future/default indicators)
        df = df.drop(columns=[c for c in self.leaky_features if c in df.columns], errors='ignore')

        print('Dropped leaky features:', [c for c in self.leaky_features if c in df])

        df = self.add_interaction_features(df) 

        # Outlier clipping + fillna
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        """for col in df.select_dtypes('number').columns:
            Q1, Q3 = df[col].quantile([0.25, 0.75])
            IQR = Q3 - Q1
            df[col] = df[col].clip(lower=Q1-1.5*IQR, upper=Q3+1.5*IQR)"""

        df.fillna(df.median(numeric_only=True), inplace=True)
        
        X = df.drop('Default_Flag', axis=1)
        y = df['Default_Flag']

        #X = self.target_encode(X, y)

        self.feature_names = X.columns.tolist()
        
        return X, y
    
    def target_encode(self, X, y=None, fit=False):
        cat_cols = [c for c in ["Payment_of_Min_Amount", "Credit_Mix", "Borrower_Tier"] if c in X.columns]

        if fit:
            encoder = ce.TargetEncoder(cols=cat_cols)
            X_encoded = encoder.fit_transform(X, y)
            self.target_encoder = encoder
        else:
            if not hasattr(self, "target_encoder"):
                raise ValueError("Target encoder not fitted. Call with fit=True first.")
            X_encoded = self.target_encoder.transform(X)
        return X_encoded
    
    def add_interaction_features(self, df):
        df = df.copy()

        df["Debt_Stress"] = df["normalized_dti"] * df["normalized_utilization"]
        df["Repayment_Stress"] = df["normalized_emi"] * df["normalized_delinquency"]
        df["Liquidity_Index"] = df["normalized_savings"] * df["normalized_emi"]
        df["Credit_Exposure"] = df["Num_Credit_Card"] * df["Credit_Utilization_Ratio"]
        df["Risk_Index"] = (
            df["normalized_dti"] + 
            df["normalized_utilization"] + 
            df["normalized_delinquency"]
        ) / 3

        return df
    
    def feature_selection(self, X, y):

        folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        importances = pd.Series(0, index=X.columns)

        for train_idx, val_idx in folds.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]

            model = XGBClassifier(n_estimators=300, max_depth=4, random_state=42)
            model.fit(X_train, y_train)

            importances += pd.Series(model.feature_importances_, index=X.columns)
        
        importances /= folds.n_splits 

        top_features = importances.sort_values(ascending=False).head(15).index

        print("Top Features: ", top_features.tolist())

        return X[top_features], top_features
    
    def objective(self, trial, X_temp, y_temp, folds):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 300, 800),
            'max_depth': trial.suggest_int('max_depth', 3, 5),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08),
            'subsample': trial.suggest_float('subsample', 0.7, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.9),
            'min_child_weight': trial.suggest_int('min_child_weight', 5, 15),
            'gamma': trial.suggest_float('gamma', 0.1, 0.5),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 1),
            'reg_lambda': trial.suggest_float('reg_lambda', 1, 5),
            'random_state': 42,
            'tree_method': 'hist',
            'eval_metric': 'auc'
        }
        
        auc_scores = []

        for train_idx, val_idx in folds.split(X_temp, y_temp):
            X_train, X_val = X_temp.iloc[train_idx].copy(), X_temp.iloc[val_idx].copy()
            y_train, y_val = y_temp.iloc[train_idx], y_temp.iloc[val_idx]

            #encoder = ce.TargetEncoder(cols=["Payment_of_Min_Amount", "Credit_Mix", "Borrower_Tier"], smoothing=10, min_samples_leaf=50)
            #X_train = encoder.fit_transform(X_train, y_train)
            #X_val = encoder.transform(X_val)

            pos = (y_temp ==1).sum()
            neg = (y_temp ==0).sum()

            scale_pos_weight = neg/ (pos + 1e-6)
            params['scale_pos_weight'] = scale_pos_weight
            
            model = XGBClassifier(**params)
            model.fit(X_train, y_train)
            y_pred = model.predict_proba(X_val)[:,1]
            auc_scores.append(roc_auc_score(y_val, y_pred))
        
        return np.mean(auc_scores)
    
    def tune_hyperparams(self, X_temp, y_temp):
        folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        study = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner(n_startup_trials=10))
        
        study.optimize(lambda trial: self.objective(trial, X_temp, y_temp, folds), n_trials=200)
        
        return study.best_params
    
    def optimize_threshold(self, y_true, y_probs):

        thresholds = np.linspace(0.1, 0.9, 50)
        best_ks = 0
        best_thresh = 0.5

        for t in thresholds:
            preds = (y_probs >= t).astype(int)

            tpr = ((preds ==1) & (y_true == 1)).sum() / (y_true ==1).sum()
            fpr = ((preds ==1) & (y_true == 0)).sum() / (y_true ==0).sum()

            ks = tpr - fpr 

            if ks > best_ks:
                best_ks = ks
                best_thresh = t 

        self.best_threshold = best_thresh

        return best_thresh
    
    def train(self, filepath):
        print('Loading data...')
        X, y = self.load_and_preprocess(filepath)
        
        #print('Feature selection...')
        #X_selected, self.feature_names = self.feature_selection(X, y)
        
        print('Splitting...')
        X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

        X_temp = self.target_encode(X_temp, y_temp, fit=True)
        X_test = self.target_encode(X_test, fit=False)

        print("Feature Selection...")
        X_temp, selected_features = self.feature_selection(X_temp, y_temp)
        X_test = X_test[selected_features]

        self.feature_names = selected_features
        
        print('Tuning hyperparams...')
        best_params = self.tune_hyperparams(X_temp, y_temp)
        print('Best params:', best_params)

        X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=42)

        self.model = XGBClassifier(**best_params, early_stopping_rounds=50, random_state=42, tree_method="hist", grow_policy="lossguide", max_leaves=32)

        self.model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        y_probs = self.model.predict_proba(X_val)[:,1]
        best_thresh = self.optimize_threshold(y_val, y_probs)

        print(f"OPtimal Threshold: {best_thresh:.3f}")        
        test_auc = roc_auc_score(y_test, self.model.predict_proba(X_test)[:,1])
        
        print(f'Test AUC: {test_auc:.4f}')
        
        if test_auc >= 0.85:
            print('🎉 Target AUC 0.85 achieved!')
        else:
            print('Target not met, consider ensemble/more features.')
        
        return test_auc
    
    def save(self, path='kinga_model.pkl'):
        joblib.dump(self, path)
        
        print(f'Model saved to {path}')

    def predict_proba(self, X_new):
        X_new = self.add_interaction_features(X_new)

        X_new = self.target_encoder.transform(X_new)

        X_new = X_new[self.feature_names]
        
        return self.model.predict_proba(X_new)[:,1]
        

if __name__ == '__main__':
    model = KingaMetricCreditRiskModel()
    auc = model.train('datasets/kingametric_credit_risk.csv')
    
    model.save()

Loading data...
Dropped leaky features: []
Splitting...
Feature Selection...


[I 2026-03-24 09:23:59,465] A new study created in memory with name: no-name-ca7548d7-a176-4e9b-9b0d-c895705dd960


Top Features:  ['Borrower_Tier', 'normalized_delinquency', 'Risk_Index', 'normalized_emi', 'Payment_Instability', 'Num_of_Loan', 'Repayment_Stress', 'Total_EMI_per_month', 'Liquidity_Index', 'population_density_factor', 'Changed_Credit_Limit', 'Monthly_Inhand_Salary', 'Num_Credit_Inquiries', 'Annual_Income', 'Amount_invested_monthly']
Tuning hyperparams...


[I 2026-03-24 09:24:01,744] Trial 0 finished with value: 0.6256977531335933 and parameters: {'n_estimators': 651, 'max_depth': 3, 'learning_rate': 0.042204695607786266, 'subsample': 0.7062154231922427, 'colsample_bytree': 0.804081538349238, 'min_child_weight': 14, 'gamma': 0.4931340886779294, 'reg_alpha': 0.9847491378400525, 'reg_lambda': 1.8395929280301102}. Best is trial 0 with value: 0.6256977531335933.
[I 2026-03-24 09:24:04,362] Trial 1 finished with value: 0.6039155875145 and parameters: {'n_estimators': 697, 'max_depth': 5, 'learning_rate': 0.07596582199636151, 'subsample': 0.8228754396905571, 'colsample_bytree': 0.73848561307778, 'min_child_weight': 10, 'gamma': 0.22727083527963152, 'reg_alpha': 0.8751094994921813, 'reg_lambda': 4.735184297821082}. Best is trial 0 with value: 0.6256977531335933.
[I 2026-03-24 09:24:05,950] Trial 2 finished with value: 0.6163227575059229 and parameters: {'n_estimators': 649, 'max_depth': 3, 'learning_rate': 0.07214558192489694, 'subsample': 0.87

Best params: {'n_estimators': 309, 'max_depth': 3, 'learning_rate': 0.022300866974570996, 'subsample': 0.7396704843563581, 'colsample_bytree': 0.7213092975447193, 'min_child_weight': 6, 'gamma': 0.10008586260168865, 'reg_alpha': 0.7706051942497405, 'reg_lambda': 3.8787213617264626}
OPtimal Threshold: 0.329
Test AUC: 0.6357
Target not met, consider ensemble/more features.
Model saved to kinga_model.pkl
